# APEX Optimizer Tutorial: Math Problem Solving

This tutorial demonstrates how to use **APEX** (Analysis-based Prompt Engineering eXpert) to improve GPT-5 Mini's performance on AIME math problems through systematic prompt optimization.

APEX analyzes failures, recognizes success patterns, generates hypotheses, and validates improvements empirically.

## Configuration

All modifiable parameters in one place for easy adjustment:

In [1]:
# API Configuration
api_key = 'sk-12345'# Will prompt if not set
base_url = 'https://nexus-master.lmndstaging.com'

# Student Model Configuration (model being optimized)
student_model = "litellm_proxy/openai/gpt-5-mini"
student_base_url = base_url  # Optional custom API endpoint
student_temperature = 0.0  # Deterministic for math
student_reasoning_effort = 'minimal'

# Analysis Model Configuration (for failure analysis and hypotheses)
analysis_model = "litellm_proxy/openai/gpt-5"
analysis_base_url = base_url  # Optional custom API endpoint
analysis_temperature = 1.0  # Creative for hypothesis generation
analysis_reasoning_effort = 'minimal'

# APEX Optimization Settings
max_iterations = 10
num_hypotheses = 1
train_sample_size = 20
success_threshold = 1.0
convergence_patience = 3
num_threads = 50
seed = 42

# MLflow Tracking (Optional)
use_mlflow = True
mlflow_tracking_uri = "http://localhost:5005"
mlflow_experiment_name = "APEX-AIME-Math"

## Setup

Import dependencies and configure language models:

In [2]:
import os
import dspy
from dspy.adapters import JSONAdapter

if api_key is None:
    api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")

# Configure student model
student_kwargs = {
    "model": student_model,
    "api_key": api_key,
    "temperature": student_temperature,
}
if student_base_url:
    student_kwargs["base_url"] = student_base_url
if student_reasoning_effort:
    student_kwargs["reasoning_effort"] = student_reasoning_effort

student_lm = dspy.LM(**student_kwargs)

# Configure analysis model
analysis_kwargs = {
    "model": analysis_model,
    "api_key": api_key,
    "temperature": analysis_temperature,
}
if analysis_base_url:
    analysis_kwargs["base_url"] = analysis_base_url
if analysis_reasoning_effort:
    analysis_kwargs["reasoning_effort"] = analysis_reasoning_effort

analysis_lm = dspy.LM(**analysis_kwargs)

analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=num_threads)

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

Load AIME problems (American Invitational Mathematics Examination):

In [3]:
from datasets import load_dataset
import random

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

train_set, val_set, test_set = init_dataset()

print(f"Training: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

Training: 45 | Validation: 45 | Test: 150


Example problem:

In [4]:
print("Problem:", train_set[0]['problem'])
print("\nAnswer:", train_set[0]['answer'])

Problem: In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.

Answer: 242


## Program

Define a Chain of Thought program:

In [5]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

## Metrics

Define evaluation metrics:

In [6]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [7]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer. "
            f"You responded with '{prediction.answer}', which couldn't be parsed. "
            f"The correct answer is '{correct_answer}'."
        )
        
        if written_solution:
            feedback_text += f" Here's the full solution:\n{written_solution}"
        
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    
    if score == 1:
        feedback_text = f"Correct! The answer is '{correct_answer}'."
    else:
        feedback_text = f"Incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full solution:\n{written_solution}"

    return dspy.Prediction(score=score, feedback=feedback_text)

## Baseline Evaluation

Evaluate the unoptimized program:

In [8]:
eval_kwargs = dict(
    num_threads=num_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

print("Evaluating baseline...")
baseline_result = evaluate(program)

print(f"\nBaseline Performance: {baseline_result.score / 100.:.1%}")

Evaluating baseline...
Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:00<00:00, 160.30it/s]

2025/10/16 23:34:41 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]



Baseline Performance: 53.3%


## APEX Optimization

Optimize the program with APEX:

In [9]:
from dspy.teleprompt.apex import APEX

optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,
    hypothesis_lm=analysis_lm,
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=max_iterations,
    num_hypotheses=num_hypotheses,
    num_eval_runs=1,
    train_sample=train_sample_size,
    success_threshold=success_threshold,
    convergence_patience=convergence_patience,
    num_threads=num_threads,
    verbosity="high",
    seed=seed,
    use_mlflow=use_mlflow,
    mlflow_tracking_uri=mlflow_tracking_uri,
    mlflow_experiment_name=mlflow_experiment_name,
)

print("Starting optimization...")
optimized_program = optimizer.compile(
    student=program,
    trainset=train_set,
    valset=val_set,
)

print("\nOptimization complete!")

2025/10/16 23:34:41 INFO dspy.teleprompt.apex.apex: APEX: MLflow tracking enabled


Starting optimization...


2025/10/16 23:34:41 INFO dspy.teleprompt.apex.apex: APEX: running with num_threads=50
2025/10/16 23:34:41 INFO dspy.teleprompt.apex.apex: APEX: Configuration - max_iterations=10, num_hypotheses=1, success_threshold=1.00, convergence_patience=3
2025/10/16 23:34:41 INFO dspy.teleprompt.apex.apex: APEX: Using seed=42 for reproducibility
2025/10/16 23:34:41 INFO dspy.teleprompt.apex.apex: APEX: Evaluating initial baseline on validation set


Processed 45 / 45 examples: 100%|██████████| 45/45 [00:01<00:00, 43.60it/s]

2025/10/16 23:34:42 INFO dspy.teleprompt.apex.apex: APEX: Initial baseline score=0.5111


...

Processed 45 / 45 examples: 100%|██████████| 45/45 [00:00<00:00, 59.65it/s]

2025/10/16 23:34:53 INFO dspy.teleprompt.apex.apex: APEX: iteration 5 hypothesis score=0.5778
2025/10/16 23:34:53 INFO dspy.teleprompt.apex.apex: APEX: hypothesis details → {'observation': "Recurring failures stem from unchecked restrictive assumptions (e.g., 'rectangles require antipodal pairs', 'every row/column used'), missed alternative-valid configurations, and occasional violation of the final-answer contract (answer='0' or wrong integer despite near-complete setup). Success cases consistently follow a stepwise derivation and respect a bare-integer final output.", 'fixable_root_causes': ['Ambiguous/over-restrictive assumptions not explicitly justified', 'Missing exploration of non-diameter/non-full-usage configurations', 'Final answer contract not strictly enforced on last line'], 'non_fixable_root_causes': [], 'impact_score': 0.78, 'generalizability_score': 0.82, 'strategy': 'Add a minimal but explicit Assumption Justification + Counterexample Probe gate right before computation

2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: iteration 5 best score=0.6222
2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: Score improvements from baseline: [-0.04444444444444451]
2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: No improvement (3/3 patience)
2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: Stopping due to convergence patience reached
2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: Optimization complete - stopped after 5 iterations (patience)
2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: Final score: 0.6222 (initial baseline: 0.5111)
2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: Summary - evaluated 10 candidates from 5 hypotheses
2025/10/16 23:34:54 INFO dspy.teleprompt.apex.apex: APEX: Best score trajectory across iterations: [0.6, 0.6222222222222222, 0.6222222222222222, 0.6222222222222222, 0.6222222222222222]


🏃 View run adventurous-ray-712 at: http://localhost:5005/#/experiments/2/runs/d1a197d09b8b4df3928a2f2265bb78d3
🧪 View experiment at: http://localhost:5005/#/experiments/2

Optimization complete!


[Trace(trace_id=tr-b644a6edc6f06257a2f8401f4579fbfd), Trace(trace_id=tr-40617de67211f98b8985f36808c73eeb), Trace(trace_id=tr-153409c3e393bf5aa1931e635a148e8a), Trace(trace_id=tr-d4b2bf90481cf42dc065498270d0aeac), Trace(trace_id=tr-8270a115a0e68302a9b8e35976ce9f23), Trace(trace_id=tr-bde0475026ad5324be0e3c612b323fe4), Trace(trace_id=tr-d1b1c0c3e009be71a18c2b1e20db6239), Trace(trace_id=tr-182e54bf06ee3e159ee09843661ce34e), Trace(trace_id=tr-0b3fcfe8035316a87b945a2b1809473e), Trace(trace_id=tr-19d8e1ac0e59b50605ecf57e2a41ee0d)]

Inspect the optimized prompt:

In [10]:
print("Optimized Prompt:")
print("=" * 50)
print(optimized_program.predict.signature.instructions)
print("=" * 50)

Optimized Prompt:
You are a careful olympiad-level problem solver. Solve the given math problem with rigorous control of assumptions and a strict final output contract.

Protocol:
1) Parse & List Constraints: Identify all given conditions, domains (integers/rationals/reals), symmetries, and objective (count/optimize/compute exact value). Note any format requirement for the final answer (bare integer).
2) Plan: Outline a short plan (1–3 sentences) indicating the approach (e.g., combinatorics casework, modular residues, generating functions, geometric relations, algebraic substitutions).
3) Assumption Audit (Gate A):
   - List each simplifying assumption you intend to use (e.g., “only antipodal pairs”, “every row/column used”). For each, state why it must hold. If it is not logically forced by the problem, mark it as tentative.
   - Counterexample Probe: For the strongest tentative assumption, actively try to construct a configuration that satisfies all problem constraints while violatin

## Final Evaluation

Evaluate the optimized program:

In [11]:
print("Evaluating optimized program...")
optimized_result = evaluate(optimized_program)

print(f"\n{'='*50}")
print(f"Baseline:  {baseline_result.score/100.:.1%}")
print(f"Optimized: {optimized_result.score/100.:.1%}")
print(f"Improvement: {(optimized_result.score - baseline_result.score)/100.:.1%}")
print(f"{'='*50}")

Evaluating optimized program...
Average Metric: 105.00 / 150 (70.0%): 100%|██████████| 150/150 [00:00<00:00, 526.33it/s]

2025/10/16 23:34:54 INFO dspy.evaluate.evaluate: Average Metric: 105 / 150 (70.0%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,1) Parse & List Constraints: - We're given positional numeral repr...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,1) Parse & List Constraints: - Triangle ABC with points on AB in o...,588,✔️ [1]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,1) Parse & List Constraints: - There are 9 distinct players. Each ...,16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"1) Parse & List Constraints: - Variables x,y are integers with -10...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,1) Parse & List Constraints: - We consider all 8-digit numbers tha...,279,✔️ [1]



Baseline:  53.3%
Optimized: 70.0%
Improvement: 16.7%


## Optimization Insights

Examine the optimization process:

In [12]:
if hasattr(optimized_program, 'apex_result'):
    result = optimized_program.apex_result
    
    print("Summary:")
    print(f"  Iterations: {len(result.iterations)}")
    print(f"  Candidates evaluated: {len(result.all_candidates)}")
    print(f"  Stop reason: {result.stopped_after}")
    print(f"  Best score: {result.best_candidate.overall_score:.4f}")
    
    print("\nIteration Progress:")
    for it in result.iterations:
        print(f"  Iteration {it.iteration}: {it.num_failures} failures, {len(it.hypotheses)} hypotheses, {len(it.candidates)} candidates")
    
    if result.best_candidate.hypothesis:
        h = result.best_candidate.hypothesis
        print(f"\nBest Hypothesis:")
        print(f"  Strategy: {h.strategy if hasattr(h, 'strategy') else 'N/A'}")
        print(f"  Impact Score: {h.impact_score if hasattr(h, 'impact_score') else 'N/A'}")

Summary:
  Iterations: 5
  Candidates evaluated: 11
  Stop reason: patience
  Best score: 0.6222

Iteration Progress:
  Iteration 1: 6 failures, 1 hypotheses, 2 candidates
  Iteration 2: 6 failures, 1 hypotheses, 2 candidates
  Iteration 3: 6 failures, 1 hypotheses, 2 candidates
  Iteration 4: 20 failures, 1 hypotheses, 2 candidates
  Iteration 5: 6 failures, 1 hypotheses, 2 candidates

Best Hypothesis:
  Strategy: Tighten the existing solution protocol into a short, enforceable checklist with required gates (Derive → Validate assumptions → Constraint audit → Independent numeric cross-check → Numeric-only output). Add explicit DO-NOT behaviors (no guessing, no unverified heuristics) and a compact template for the reasoning to prevent skipping steps, while preserving the successful stepwise derivations seen in correct cases.
  Impact Score: 0.8


## Conclusion

APEX systematically optimizes prompts through:
1. Analyzing failures to understand root causes
2. Recognizing successful patterns
3. Generating data-driven hypotheses
4. Validating improvements empirically

Try adjusting the configuration parameters to explore different optimization strategies.